<a href="https://colab.research.google.com/github/awildt01/Credit-Scoring/blob/main/notebooks/7_LGD_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Der **Loss Given Default (LGD)** ist ein Schlüsselparameter im Kreditrisikomanagement, der angibt, wie viel ein Kreditgeber voraussichtlich verlieren wird, wenn ein Kreditnehmer ausfällt. Er wird typischerweise als Prozentsatz des ausstehenden Kreditbetrags zum Zeitpunkt des Ausfalls ausgedrückt (1 - Recovery Rate).

**Der Two-Step-Modellansatz** (oder Zweistufenmodell) ist eine gängige Methode zur Modellierung des LGD, insbesondere wenn ein erheblicher Anteil der ausgefallenen Kredite keine Rückgewinnung erfährt (d.h. die Recovery Rate 0% beträgt).

Dieser Ansatz unterteilt den Modellierungsprozess in zwei Schritte:

**Schritt 1: Modellierung der Wahrscheinlichkeit einer Null-Rückgewinnung (Probability of Zero Recovery - PZR)**

- **Ziel:** Vorhersagen, ob überhaupt eine Rückgewinnung stattfinden wird oder ob der gesamte ausstehende Betrag verloren geht (Recovery Rate = 0%).

- **Abhängige Variable:** Eine binäre Variable, wie die recovery_rate_0_1, die wir erstellt haben. Sie ist 1, wenn es eine positive Rückgewinnung gab, und 0, wenn die Rückgewinnung 0 war.

- **Modelltyp:** Typischerweise wird hier ein binäres Klassifikationsmodell verwendet, wie eine logistische Regression oder ein Entscheidungsbaum. Dieses Modell schätzt die Wahrscheinlichkeit, dass ein ausgefallener Kredit eine Rückgewinnung von Null haben wird.

**Schritt 2: Modellierung der Rückgewinnungshöhe (Conditional Recovery Rate)**

- **Ziel:** Für die Fälle, in denen eine Rückgewinnung erwartet wird (d.h. das Modell aus Schritt 1 hat eine positive Rückgewinnung vorhergesagt), wird die tatsächliche Höhe der Recovery Rate modelliert.

- **Abhängige Variable:** Die recovery_rate selbst, aber nur für die Fälle, in denen sie größer als 0 ist (oder allgemeiner, zwischen 0 und 1 liegt, exklusive 0).

- **Modelltyp:** Hier kommen in der Regel Regressionsmodelle zum Einsatz, wie zum Beispiel die lineare Regression, Beta-Regression oder TOBIT-Modelle (die zensierte Daten berücksichtigen können, da die Recovery Rate auf 0 und 1 begrenzt ist).

**Warum dieser zweistufige Ansatz?**

1. **Umgang mit Zensierung:** Die Recovery Rate ist auf den Bereich [0, 1] zensiert (begrenzt). Ein hoher Anteil von 0-Rückgewinnungen kann die Schätzung eines einzelnen Regressionsmodells verzerren, da es Schwierigkeiten hat, den Unterschied zwischen einem echten Nullwert und einem Wert, der nahe Null, aber positiv ist, zu handhaben.

2. **Bessere Modellierung der Realität:** Der Ansatz spiegelt oft die Realität besser wider. Es gibt einen qualitativen Unterschied zwischen gar keiner Rückgewinnung und einer geringen Rückgewinnung. Die Gründe dafür können sehr unterschiedlich sein (z.B. keine Sicherheiten vs. Wertverfall von Sicherheiten).

3. **Verbesserte Vorhersagegenauigkeit:** Durch die separate Modellierung der beiden Aspekte (ob eine Rückgewinnung stattfindet und wie hoch sie ist) kann die Gesamtgenauigkeit der LGD-Schätzung oft verbessert werden.


Zusammenfassend lässt sich sagen, dass der Two-Step-Modellansatz für LGD eine robuste Methode ist, um die Komplexität der Recovery Rate zu handhaben, insbesondere bei der Berücksichtigung von Fällen mit Null-Rückgewinnungen, und führt in der Regel zu präziseren und stabileren LGD-Schätzungen.

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np


# Der Pfad zu deiner Datei im Drive
file_path = '/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/loan_data_defaults.csv'

# Datei einlesen
loan_data_defaults = pd.read_csv(file_path)

# Kurze Bestätigung: Die ersten 5 Zeilen anzeigen
print("Datei erfolgreich aus Drive geladen!")
display(loan_data_defaults.head())


Datei erfolgreich aus Drive geladen!


/tmp/ipython-input-1032701199.py:9: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  loan_data_defaults = pd.read_csv(file_path)


,Unnamed: 0.1,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,...,addr_state:WA,addr_state:WI,addr_state:WV,addr_state:WY,initial_list_status:f,initial_list_status:w,good_bad,recovery_rate,CCF,recovery_rate_0_1
0,1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,...,0,0,0,0,1,0,0,0.046832,0.817416,1
1,8,8,1071795,1306957,5600,5600,5600.0,60 months,21.28,152.39,...,0,0,0,0,1,0,0,0.033761,0.971068,1
2,9,9,1071570,1306721,5375,5375,5350.0,60 months,12.69,121.45,...,0,0,0,0,1,0,0,0.050100,0.874701,1
3,12,12,1064687,1298717,9000,9000,9000.0,36 months,13.49,305.38,...,0,0,0,0,1,0,0,0.049367,0.860429,1
4,14,14,1069057,1303503,10000,10000,10000.0,36 months,10.65,325.74,...,0,0,0,0,1,0,0,0.064510,0.456653,1


# LGD Model

In [5]:
from sklearn.model_selection import train_test_split

### Splitting Data

Die Spalten 'good_bad', 'recovery_rate', 'recovery_rate_0_1' und 'CCF' werden aus den Eingabefeatures für das Modell lgd_inputs_stage_1 gelöscht, und zwar aus den folgenden Gründen:

**`1.good_bad:`** Diese Spalte ist ein Indikator dafür, ob ein Kreditnehmer ausgefallen ist oder nicht. Da das LGD-Modell (Loss Given Default) nur für bereits ausgefallene Kredite angewendet wird, ist good_bad keine Variable, die zur Vorhersage der Rückgewinnungsquote selbst verwendet wird. Sie ist relevant für ein PD-Modell (Probability of Default), aber nicht für das LGD-Modell.

**`2.recovery_rate:`** Dies ist die tatsächliche Rückgewinnungsquote. Sie ist die Zielvariable für den zweiten Schritt des LGD-Modells. Im ersten Schritt des Two-Step-Modells (lgd_targets_stage_1) geht es jedoch darum, vorherzusagen, ob überhaupt eine Rückgewinnung stattfindet (0 oder 1). Wenn 'recovery_rate' als Feature in diesem ersten Schritt verwendet würde, wäre das ein Target Leakage, da sie direkt die Information enthält, die das Modell vorhersagen soll.

**`3.recovery_rate_0_1:`** Dies ist die Zielvariable für den ersten Schritt des LGD-Modells (wie im Code loan_data_defaults['recovery_rate_0_1'] als Target zugewiesen). Eine Zielvariable darf niemals gleichzeitig als Eingabefeature verwendet werden, da dies ebenfalls zu Target Leakage führen und das Modell unrealistisch gut erscheinen lassen würde.

**`4.CCF (Credit Conversion Factor):`** Der CCF ist ein Parameter, der typischerweise im EAD-Modell (Exposure at Default) verwendet wird, um den Betrag eines Kredits zu schätzen, der zum Zeitpunkt des Ausfalls tatsächlich gezogen wurde. Er ist nicht direkt relevant für die Vorhersage der Rückgewinnungsquote oder ob eine Rückgewinnung stattfinden wird. Daher wird er aus den Features entfernt.

In [6]:
lgd_inputs_stage_1_train, lgd_inputs_stage_1_test, lgd_targets_stage_1_train, lgd_targets_stage_1_test = train_test_split(loan_data_defaults.drop(['good_bad', 'recovery_rate', 'recovery_rate_0_1', 'CCF'], axis = 1), loan_data_defaults['recovery_rate_0_1'], test_size = 0.2, random_state = 42)

In [7]:
features_all = ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:MORTGAGE',
'home_ownership:NONE',
'home_ownership:OTHER',
'home_ownership:OWN',
'home_ownership:RENT',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:car',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:educational',
'purpose:home_improvement',
'purpose:house',
'purpose:major_purchase',
'purpose:medical',
'purpose:moving',
'purpose:other',
'purpose:renewable_energy',
'purpose:small_business',
'purpose:vacation',
'purpose:wedding',
'initial_list_status:f',
'initial_list_status:w',
'term_int',
'emp_length_int',
'mths_since_issue_d',
'mths_since_earliest_cr_line',
'funded_amnt',
'int_rate',
'installment',
'annual_inc',
'dti',
'delinq_2yrs',
'inq_last_6mths',
'mths_since_last_delinq',
'mths_since_last_record',
'open_acc',
'pub_rec',
'total_acc',
'acc_now_delinq',
'total_rev_hi_lim']

In [8]:
features_reference_cat = ['grade:G',
'home_ownership:RENT',
'verification_status:Verified',
'purpose:credit_card',
'initial_list_status:f']

In [9]:
lgd_inputs_stage_1_train = lgd_inputs_stage_1_train[features_all]

In [10]:
lgd_inputs_stage_1_train = lgd_inputs_stage_1_train.drop(features_reference_cat, axis = 1)

In [11]:
# fehlende werte berechnen
lgd_inputs_stage_1_train.isnull().sum()

,0
grade:A,0
grade:B,0
grade:C,0
grade:D,0
grade:E,0
grade:F,0
home_ownership:MORTGAGE,0
home_ownership:NONE,0
home_ownership:OTHER,0
home_ownership:OWN,0


In [12]:
lgd_inputs_stage_1_train['purpose:moving'].head()

,purpose:moving
19698,0
26556,0
21857,0
9905,0
9205,0


### Estimating the Model

In [13]:
from sklearn import linear_model
import scipy.stats as stat

class LogisticRegression_with_p_values:

    def __init__(self,*args,**kwargs):
        # Erhöhen Sie max_iter, um die Konvergenz zu ermöglichen
        self.model = linear_model.LogisticRegression(*args,max_iter=1000,C=1.0,solver='liblinear'
        ,penalty='l2',**kwargs)


    def fit(self,X,y):
        self.model.fit(X,y)
        denom = (2.0 * (1.0 + np.cosh(self.model.decision_function(X))))
        denom = np.tile(denom,(X.shape[1],1)).T
        F_ij = np.dot((X / denom).T,X)
        Cramer_Rao = np.linalg.inv(F_ij)
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores]
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values

In [14]:
reg_lgd_st_1 = LogisticRegression_with_p_values()
reg_lgd_st_1.fit(lgd_inputs_stage_1_train, lgd_targets_stage_1_train)

# Warning: ConvergenceWarning
Um die `ConvergenceWarning` zu beheben und die Modellstabilität zu verbessern, werde ich die Trainings- und Testdaten mit `StandardScaler` skalieren, das Modell `LogisticRegression_with_p_values` mit den skalierten Daten neu trainieren und dann die aktualisierte Übersichtstabelle mit Koeffizienten und p-Werten anzeigen, um die Auswirkungen der Skalierung zu bewerten. Dies wird dazu beitragen, Merkmale mit unterschiedlichen Skalen zu standardisieren, was die Modellkonvergenz erheblich verbessern kann.

Übersetzt mit DeepL.com (kostenlose Version)

In [15]:
feature_name = lgd_inputs_stage_1_train.columns.values

In [16]:
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg_lgd_st_1.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg_lgd_st_1.intercept_[0]]
summary_table = summary_table.sort_index()
p_values = reg_lgd_st_1.p_values
p_values = np.append(np.nan,np.array(p_values))
summary_table['p_values'] = p_values
summary_table

,Feature name,Coefficients,p_values
0,Intercept,-4.036295e-03,NaN
1,grade:A,-1.259942e-03,9.882968e-01
2,grade:B,-4.183812e-03,9.520837e-01
3,grade:C,-4.651505e-03,9.447506e-01
4,grade:D,1.192433e-03,9.863819e-01
5,grade:E,1.251371e-03,9.870390e-01
6,grade:F,3.029629e-03,9.727096e-01
7,home_ownership:MORTGAGE,-3.241695e-04,9.899112e-01
8,home_ownership:NONE,2.768559e-05,9.999728e-01
9,home_ownership:OTHER,-5.647526e-05,9.998972e-01


## Skalieren der Trainingsdaten

### Teilaufgabe::
Hier wird  ein `StandardScaler` auf die Trainingsdaten `lgd_inputs_stage_1_train` angewendet. Dies hilft, Merkmale mit unterschiedlichen Skalen zu standardisieren, was die Konvergenz des Modells verbessern kann.


**Begründung:** Zunächst importiere ich den `StandardScaler` aus `sklearn.preprocessing`, instanziieren ihn und passe dann den DataFrame `lgd_inputs_stage_1_train` an und transformiere ihn, um die Merkmale zu standardisieren.



In [17]:
from sklearn.preprocessing import StandardScaler

# Instantiate a StandardScaler object
scaler = StandardScaler()

# Fit the scaler to the training data and transform it
lgd_inputs_stage_1_train_scaled = scaler.fit_transform(lgd_inputs_stage_1_train)

# Convert the scaled data back to a DataFrame, retaining column names
lgd_inputs_stage_1_train_scaled = pd.DataFrame(lgd_inputs_stage_1_train_scaled, columns=lgd_inputs_stage_1_train.columns)

print("Training data scaled successfully!")
display(lgd_inputs_stage_1_train_scaled.head())

Training data scaled successfully!


,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:MORTGAGE,home_ownership:NONE,home_ownership:OTHER,home_ownership:OWN,...,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,total_acc,acc_now_delinq,total_rev_hi_lim
0,-0.250483,-0.522291,-0.620631,1.831537,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,1.043233,-0.347487,0.722766,-0.675239,-0.3398,1.231901,-0.317239,0.864364,-0.052635,-0.433977
1,-0.250483,-0.522291,1.611265,-0.545990,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,1.158302,-0.347487,1.492985,-0.675239,-0.3398,0.006566,-0.317239,-0.962346,-0.052635,-0.946586
2,-0.250483,-0.522291,-0.620631,1.831537,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,0.278664,-0.347487,-0.047454,-0.675239,-0.3398,-0.810324,-0.317239,1.212309,-0.052635,-0.390860
3,-0.250483,-0.522291,1.611265,-0.545990,-0.396308,-0.263336,1.122131,-0.014228,-0.031827,-0.309637,...,-0.107456,-0.347487,-0.817674,-0.675239,2.1307,0.210788,4.313558,0.516420,-0.052635,-0.812445
4,-0.250483,-0.522291,-0.620631,-0.545990,2.523289,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,0.783689,-0.347487,-0.817674,2.524660,-0.3398,0.415011,-0.317239,0.951351,-0.052635,-0.328581


**Begründung**:
Nach der Skalierung der Trainingsdaten ist es entscheidend, die Testdaten mit dem *gleichen* angepassten `StandardScaler` zu skalieren, um Konsistenz zu gewährleisten und Datenlecks zu verhindern. Damit werden die Testdaten für die Modellbewertung vorbereitet.



In [18]:
lgd_inputs_stage_1_test_scaled = scaler.transform(lgd_inputs_stage_1_test[lgd_inputs_stage_1_train.columns])

# Convert the scaled test data back to a DataFrame
lgd_inputs_stage_1_test_scaled = pd.DataFrame(lgd_inputs_stage_1_test_scaled, columns=lgd_inputs_stage_1_train.columns)

print("Test data scaled successfully using the fitted scaler!")
display(lgd_inputs_stage_1_test_scaled.head())

Test data scaled successfully using the fitted scaler!


,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:MORTGAGE,home_ownership:NONE,home_ownership:OTHER,home_ownership:OWN,...,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,total_acc,acc_now_delinq,total_rev_hi_lim
0,-0.250483,-0.522291,-0.620631,1.831537,-0.396308,-0.263336,1.122131,-0.014228,-0.031827,-0.309637,...,0.452546,-0.347487,1.492985,1.724685,-0.339800,-0.197657,-0.317239,0.081489,-0.052635,-0.687886
1,3.992280,-0.522291,-0.620631,-0.545990,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,-1.329743,-0.347487,-0.817674,-0.675239,-0.339800,-1.218770,-0.317239,-1.310291,-0.052635,-0.822027
2,-0.250483,-0.522291,-0.620631,-0.545990,-0.396308,3.797423,-0.891162,-0.014228,-0.031827,-0.309637,...,0.490903,0.938504,-0.817674,-0.141923,3.184295,-0.401879,1.998159,-0.701387,-0.052635,-0.041136
3,3.992280,-0.522291,-0.620631,-0.545990,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,-0.309637,...,-0.728827,-0.347487,-0.817674,-0.675239,-0.339800,0.210788,-0.317239,0.255461,-0.052635,-0.127370
4,-0.250483,-0.522291,-0.620631,1.831537,-0.396308,-0.263336,-0.891162,-0.014228,-0.031827,3.229593,...,0.173824,-0.347487,0.722766,0.924710,-0.339800,1.844569,-0.317239,0.690392,-0.052635,2.761447


**Begründung**:
Nachdem nun sowohl die Trainings- als auch die Testdaten skaliert wurden, besteht der nächste logische Schritt darin, das logistische Regressionsmodell unter Verwendung der skalierten Trainingsdaten neu zu trainieren. Dies wird dazu beitragen, die `ConvergenceWarning` zu beheben und die Modellstabilität zu verbessern, wie in der Hauptaufgabe beschrieben.


In [19]:
reg_lgd_st_1_scaled = LogisticRegression_with_p_values()
reg_lgd_st_1_scaled.fit(lgd_inputs_stage_1_train_scaled, lgd_targets_stage_1_train)
print("Logistic Regression model retrained with scaled data.")

Logistic Regression model retrained with scaled data.


In [20]:
feature_name_scaled = lgd_inputs_stage_1_train_scaled.columns.values
summary_table_scaled = pd.DataFrame(columns = ['Feature name'], data = feature_name_scaled)
summary_table_scaled['Coefficients'] = np.transpose(reg_lgd_st_1_scaled.coef_)
summary_table_scaled.index = summary_table_scaled.index + 1
summary_table_scaled.loc[0] = ['Intercept', reg_lgd_st_1_scaled.intercept_[0]]
summary_table_scaled = summary_table_scaled.sort_index()
p_values_scaled = reg_lgd_st_1_scaled.p_values
p_values_scaled = np.append(np.nan,np.array(p_values_scaled))
summary_table_scaled['p_values'] = p_values_scaled
print("\n**Zusammenfassungstabelle mit StandardScaler:**")
display(summary_table_scaled)


**Zusammenfassungstabelle mit StandardScaler:**


,Feature name,Coefficients,p_values
0,Intercept,0.279237,NaN
1,grade:A,0.532498,4.266107e-36
2,grade:B,0.694231,7.921264e-31
3,grade:C,0.571489,3.038136e-24
4,grade:D,0.421379,7.456138e-20
5,grade:E,0.192293,1.606285e-08
6,grade:F,0.089786,2.144150e-04
7,home_ownership:MORTGAGE,-0.001778,8.919611e-01
8,home_ownership:NONE,0.016477,2.880119e-01
9,home_ownership:OTHER,-0.016466,1.760607e-01


In [22]:
display(summary_table)

,Feature name,Coefficients,p_values
0,Intercept,-4.036295e-03,NaN
1,grade:A,-1.259942e-03,9.882968e-01
2,grade:B,-4.183812e-03,9.520837e-01
3,grade:C,-4.651505e-03,9.447506e-01
4,grade:D,1.192433e-03,9.863819e-01
5,grade:E,1.251371e-03,9.870390e-01
6,grade:F,3.029629e-03,9.727096e-01
7,home_ownership:MORTGAGE,-3.241695e-04,9.899112e-01
8,home_ownership:NONE,2.768559e-05,9.999728e-01
9,home_ownership:OTHER,-5.647526e-05,9.998972e-01


### Vergleich und Auswirkungen der Datenskalierung:

1.  **Konvergenz des Modells:**
    *   **Ohne StandardScaler:** Das ursprüngliche Modell zeigte eine `ConvergenceWarning` (`STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT`), selbst nachdem `max_iter` auf 1.000.000 erhöht wurde. Dies deutet darauf hin, dass der Optimierungsalgorithmus Schwierigkeiten hatte, ein stabiles Minimum der Kostenfunktion zu finden, da die Features unterschiedliche Skalen aufwiesen.
    *   **Mit StandardScaler:** Nachdem die Daten mit dem `StandardScaler` skaliert wurden, wurde das Modell *ohne Konvergenz-Warnung* trainiert. Dies ist der wichtigste Erfolg der Skalierung: Das Modell konnte nun erfolgreich konvergieren, was auf eine stabilere und zuverlässigere Schätzung der Koeffizienten hindeutet.

2.  **Koeffizienten:**
    *   **Ohne StandardScaler:** Die Koeffizientenwerte variieren stark in ihrer Größenordnung. Zum Beispiel hat `mths_since_issue_d` einen Koeffizienten von `2.62e-02`, während `total_rev_hi_lim` einen von `7.49e-07` hat. Dies ist zu erwarten, wenn Features sehr unterschiedliche Skalen haben.
    *   **Mit StandardScaler:** Die Koeffizienten der skalierten Daten sind in der Regel vergleichbarer in ihrer Größenordnung. Dies liegt daran, dass der Scaler alle Features auf eine ähnliche Skala transformiert (Mittelwert 0, Standardabweichung 1). Während die absoluten Werte der Koeffizienten unterschiedlich sind, bleiben die Vorzeichen und die relative Wichtigkeit (gemessen an der absoluten Größe der Koeffizienten) in vielen Fällen erhalten oder werden klarer.

3.  **p-Werte und Statistische Signifikanz:**
    *   **Ohne StandardScaler:** Bereits ohne Skalierung zeigen viele Features statistisch signifikante p-Werte (z.B. `verification_status:Not Verified`, `initial_list_status:w`, `mths_since_issue_d`, `dti`). Die Konvergenzprobleme könnten jedoch die Genauigkeit dieser p-Werte beeinträchtigt haben.
    *   **Mit StandardScaler:** Die p-Werte des skalierten Modells sind nun zuverlässiger, da das Modell konvergiert ist. Auffällig ist, dass einige Features, die zuvor nicht signifikant waren (z.B. `grade:A`, `grade:B`), nun extrem niedrige p-Werte aufweisen und somit hochsignifikant werden. Dies liegt daran, dass das Modell nun in der Lage ist, die wahren Beziehungen in den Daten besser zu erfassen, da die Optimierung nicht mehr durch Skalierungsunterschiede behindert wird.

**Fazit:**

Die Anwendung des `StandardScaler` hat das entscheidende Konvergenzproblem des logistischen Regressionsmodells gelöst. Dies ist von größter Bedeutung, da ein nicht konvergierendes Modell keine zuverlässigen Koeffizienten oder p-Werte liefert. Mit den skalierten Daten sind die Modellergebnisse nun statistisch gültiger und die Interpretation der Feature-Wichtigkeit präziser. Die Skalierung hat nicht nur die Stabilität des Modells verbessert, sondern auch die statistische Signifikanz einiger Features deutlicher hervorgehoben.